In [42]:
# Loading Data from SQL
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Same DB path as Notebook 01
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH  = os.path.join(BASE_DIR, "data", "raw", "financials.db")
engine   = create_engine(f"sqlite:///{DB_PATH}")

# Load all three statements
df_income    = pd.read_sql("SELECT * FROM income",   engine)
df_balance   = pd.read_sql("SELECT * FROM balance",  engine)
df_cashflow  = pd.read_sql("SELECT * FROM cashflow", engine)

print(f"Income:   {len(df_income)} rows")
print(f"Balance:  {len(df_balance)} rows")
print(f"Cashflow: {len(df_cashflow)} rows")

# Preview available columns
print(f"\nIncome columns:\n{list(df_income.columns)}")

Income:   101 rows
Balance:  101 rows
Cashflow: 102 rows

Income columns:
['date', 'ticker', 'sector', 'company_name', 'Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Otherunder Preferred Stock Dividend', 'Net Income', 'Minority Interests', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Non Operating Income Expenses', 'Special Income Charges', 'Other Special Charges', 'Write Off', 'Impairment Of Capital Assets', 'Restructuring And Mergern Acquisition', 'Gain On Sale

In [43]:
#Merge 3 Statements
# Converting the dates
for df in [df_income, df_balance, df_cashflow]:
    df["date"] = pd.to_datetime(df["date"])

# Merging on ticker + date + sector + company_name
MERGE_ON = ["ticker", "date", "sector", "company_name"]
df = df_income.merge(df_balance,  on=MERGE_ON, suffixes=("", "_bal"))
df = df.merge(df_cashflow, on=MERGE_ON, suffixes=("", "_cf"))

print(f"Merged: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Companies: {df['company_name'].nunique()}")
print(f"Years: {df['date'].dt.year.unique()}")

Merged: 100 rows, 300 columns
Companies: 22
Years: [2024 2023 2022 2021 2025]


In [44]:
#Computing 14 ratios
# Safe division helper — returns NaN instead of crashing on divide by zero
def safe_div(a, b):
    return np.where(b != 0, a / b, np.nan)

# ── LIQUIDITY ─────────────────────────────────────────────────────────
df["current_ratio"]  = safe_div(df["Current Assets"],      df["Current Liabilities"])
df["quick_ratio"]    = safe_div(df["Current Assets"] - df["Inventory"], df["Current Liabilities"])
df["cash_ratio"]     = safe_div(df["Cash And Cash Equivalents"], df["Current Liabilities"])

# ── LEVERAGE ──────────────────────────────────────────────────────────
df["debt_to_equity"]    = safe_div(df["Total Debt"],  df["Stockholders Equity"])
df["interest_coverage"] = safe_div(df["EBIT"],        df["Interest Expense"])
df["debt_to_ebitda"]    = safe_div(df["Total Debt"],  df["EBITDA"])

# ── PROFITABILITY ─────────────────────────────────────────────────────
df["gross_margin"]   = safe_div(df["Gross Profit"],  df["Total Revenue"])
df["ebitda_margin"]  = safe_div(df["EBITDA"],        df["Total Revenue"])
df["net_margin"]     = safe_div(df["Net Income"],    df["Total Revenue"])
df["roe"]            = safe_div(df["Net Income"],    df["Stockholders Equity"])
df["roa"]            = safe_div(df["Net Income"],    df["Total Assets"])

# ── CASH FLOW ─────────────────────────────────────────────────────────
df["operating_cf_ratio"] = safe_div(df["Operating Cash Flow"],  df["Current Liabilities"])
df["cf_to_debt"]         = safe_div(df["Operating Cash Flow"],  df["Total Debt"])
df["fcf_margin"]         = safe_div(df["Free Cash Flow"],       df["Total Revenue"])

RATIO_COLS = [
    "current_ratio", "quick_ratio", "cash_ratio",
    "debt_to_equity", "interest_coverage", "debt_to_ebitda",
    "gross_margin", "ebitda_margin", "net_margin",
    "roe", "roa", "operating_cf_ratio", "cf_to_debt", "fcf_margin"
]

# Check how many non-null values we have per ratio
print("Ratio coverage (non-null values):")
for col in RATIO_COLS:
    count = df[col].notna().sum()
    print(f"  {col:25s}: {count}/{len(df)}")

Ratio coverage (non-null values):
  current_ratio            : 60/100
  quick_ratio              : 59/100
  cash_ratio               : 60/100
  debt_to_equity           : 88/100
  interest_coverage        : 72/100
  debt_to_ebitda           : 60/100
  gross_margin             : 60/100
  ebitda_margin            : 60/100
  net_margin               : 88/100
  roe                      : 88/100
  roa                      : 88/100
  operating_cf_ratio       : 60/100
  cf_to_debt               : 88/100
  fcf_margin               : 88/100


In [45]:
#Delta and z-score features
# Sort by company and date before computing trends
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

# Year-over-year change per ratio (catches deterioration like Pick n Pay)
for col in RATIO_COLS:
    df[f"delta_{col}"] = df.groupby("ticker")[col].diff()

# Sector-relative z-scores (normalises banks vs retailers vs miners)
for col in RATIO_COLS:
    df[f"zscore_{col}"] = df.groupby(["sector", "date"])[col].transform(
        lambda x: (x - x.median()) / (x.std() + 1e-6)
    )

DELTA_COLS  = [f"delta_{c}"  for c in RATIO_COLS]
ZSCORE_COLS = [f"zscore_{c}" for c in RATIO_COLS]

print(f"Delta features:  {len(DELTA_COLS)}")
print(f"Z-score features: {len(ZSCORE_COLS)}")
print(f"Total features:   {len(RATIO_COLS) + len(DELTA_COLS) + len(ZSCORE_COLS)}")



Delta features:  14
Z-score features: 14
Total features:   42


In [46]:
#Building Health SCore Labels.
def health_score(row):
    pts = 0

    # Interest coverage — single most important signal
    if   row.interest_coverage > 8:   pts += 2
    elif row.interest_coverage > 4:   pts += 1
    elif row.interest_coverage < 1.5: pts -= 3
    elif row.interest_coverage < 2.5: pts -= 1

    # Debt load
    if   row.debt_to_ebitda < 1.5: pts += 2
    elif row.debt_to_ebitda < 3.0: pts += 1
    elif row.debt_to_ebitda > 5.0: pts -= 2
    elif row.debt_to_ebitda > 4.0: pts -= 1

    # Real cash generation
    if   row.cf_to_debt > 0.25: pts += 1
    elif row.cf_to_debt < 0:    pts -= 2

    # Profitability
    if   row.ebitda_margin > 0.25: pts += 1
    elif row.ebitda_margin < 0:    pts -= 1

    # Trend signal
    if not pd.isna(row.delta_interest_coverage):
        if   row.delta_interest_coverage >  1.5: pts += 1
        elif row.delta_interest_coverage < -1.5: pts -= 1

    return max(1, min(5, 3 + pts))

# Drop rows missing the key inputs
KEY_COLS = ["interest_coverage", "debt_to_ebitda", "cf_to_debt", "ebitda_margin"]
df_clean = df.dropna(subset=KEY_COLS).copy()
df_clean["health_score"] = df_clean.apply(health_score, axis=1)

print(f"Rows after cleaning: {len(df_clean)}")
print(f"\nScore distribution:")
print(df_clean["health_score"].value_counts().sort_index())
print(f"\nLabel mapping:")
print("  1 = Distressed  2 = Weak  3 = Neutral  4 = Good  5 = Excellent")


Rows after cleaning: 60

Score distribution:
health_score
1     7
2     3
3     2
4    10
5    38
Name: count, dtype: int64

Label mapping:
  1 = Distressed  2 = Weak  3 = Neutral  4 = Good  5 = Excellent


In [47]:
#Cleaning and saving the dataset.
FEATURE_COLS = RATIO_COLS + DELTA_COLS + ZSCORE_COLS

# Start fresh from the 60-row clean dataset
df_final = df_clean.copy()

# Replace infinities with NaN
df_final = df_final.replace([np.inf, -np.inf], np.nan)

# Fill missing delta and zscore cols with 0 — don't drop rows for these
for col in DELTA_COLS + ZSCORE_COLS:
    df_final[col] = df_final[col].fillna(0)

# Winsorise — clip extreme outliers at 1st/99th percentile
for col in FEATURE_COLS:
    lo, hi = df_final[col].quantile([0.01, 0.99])
    df_final[col] = df_final[col].clip(lo, hi)

# Save to SQL
df_final.to_sql("features", engine, if_exists="replace", index=False)

# Save to CSV
processed_path = os.path.join(BASE_DIR, "data", "processed", "features.csv")
df_final.to_csv(processed_path, index=False)

print(f"Final feature table: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
print(f"Companies: {df_final['company_name'].nunique()}")
print(f"Score distribution:")
print(df_final["health_score"].value_counts().sort_index())
print(f"\nSaved to SQL and {processed_path}")


Final feature table: 60 rows, 343 columns
Companies: 15
Score distribution:
health_score
1     7
2     3
3     2
4    10
5    38
Name: count, dtype: int64

Saved to SQL and C:\Users\user\OneDrive\Desktop\Health-engine\financial-health-engine\data\processed\features.csv
